## Abstract

A reusable, cloud-scalable pipeline that pairs Cellpose-SAM segmentation with on-demand GPU compute (Modal + S3) for single-cell phenotyping of high-content microscopy, demonstrated on a *Saccharomyces cerevisiae* deletion-mutant panel.

----

:::{.callout-note title="AI usage disclosure" collapse="true"}
This is a placeholder for the AI usage disclosure. Once all authors sign the AI code form on Airtable, SlackBot will message you an AI disclosure that you should place here.
:::

## Purpose

High-content microscopy makes it easy to generate far more images than any single workstation can segment in a reasonable time. The bottleneck usually isn't the science — it's the plumbing: moving images, standing up GPUs, and getting consistent per-cell measurements out the other side. This pub shows one clean way to remove that bottleneck by combining a strong general-purpose segmentation model with elastic cloud GPU compute, so segmentation scales with your dataset instead of your hardware.

**Key takeaways:**

1. Cellpose-SAM segmentation combined with cloud GPU compute is a reliable, flexible way to scale single-cell phenotyping across large microscopy datasets.
2. The phenotyping outputs resolve gross morphological differences between strains and conditions.

**Who this is for:** Researchers and computational biologists doing high-throughput microscopy who want a segmentation and phenotyping pipeline, or who are weighing how to scale segmentation compute without building bespoke infrastructure.

**What this is _not_:** No new biology, and not a Cellpose or Modal tutorial.

**What you can do with it:** Run the accompanying notebook end-to-end on the example data, or lift the pattern (file I/O → segmentation → scale-out → per-cell measurements) into your own setup.

## How the pipeline fits together

:::{.callout-note title="Draft figure" collapse="false"}
A designed hero figure (in progress with Audrey) will replace the schematic below. The Mermaid diagram is a functional stand-in that captures the intended flow.
:::

The workflow is a straight line from raw images to per-cell measurements, with one key property: the segmentation step is identical whether it runs on one machine or hundreds of GPU containers. Scale is a deployment choice, not a code rewrite.

```{mermaid}
flowchart LR
  A["Brightfield ND2s<br/>datasets/&lt;id&gt;/raw/"] --> B["File I/O + preprocessing<br/>arcadia-microscopy-tools"]
  B --> C["Segmentation<br/>Cellpose-SAM → label masks"]
  C --> D{"Scale-out<br/>one worker per image"}
  D --> E["Derived artifacts<br/>masks · overlays · measurements"]
  E --> F["Analysis<br/>pooled per-cell morphology"]
  P["Plate map<br/>well → strain"] --> F
```

1. **Input:** brightfield ND2s in object storage, organized as `datasets/<id>/raw/`, with a plate map linking wells to strains.
2. **File I/O and preprocessing:** `arcadia-microscopy-tools` handles ND2 reading, channel and pixel-size metadata, and percentile rescaling.
3. **Segmentation:** Cellpose-SAM produces label masks; border-touching cells are dropped.
4. **Scale-out:** one worker per image is fanned across on-demand GPU containers, each reading from and writing to shared cloud storage.
5. **Outputs (`derived/`):** label-mask TIFFs, outline-overlay JPGs for QC, and per-cell measurement CSVs (regionprops, in microns where the pixel size is known).
6. **Analysis:** pooled per-cell measurements joined to the plate map for morphology comparisons across strains.

## Example dataset

The demo runs on a *Saccharomyces cerevisiae* deletion-mutant panel: single-channel brightfield ND2s of the BY4741 wild-type parent plus seven single-gene deletion mutants, imaged in two wells each.

| Strain | Wells | Role |
| --- | --- | --- |
| `by4741` | A01, A02 | WT parent |
| `thp3∆` | B01, B02 | deletion mutant |
| `rvs161∆` | C01, C02 | deletion mutant |
| `zrt3∆` | D01, D02 | deletion mutant |
| `rad6∆` | E01, E02 | deletion mutant |
| `aah1∆` | F01, F02 | deletion mutant |
| `ncr1∆` | G01, G02 | deletion mutant |
| `ppn1∆` | H01, H02 | deletion mutant |

The full well-to-strain mapping lives in `plate-map.csv` alongside this notebook. The two wells per strain are technical replicates, not independent biological replicates — a caveat we return to in the analysis and limitations.

**Acquisition (placeholder):** note the imaging modality, objective, and pixel size here, along with how wells map to ND2 filenames for this dataset.

**Where to get the data:** the example images are archived publicly so any reader can pull them (Zenodo record link to be added). Internal readers with bucket access read the same images directly from S3.

## Demo: single-cell segmentation at scale

The cells below run the full pipeline end to end: brightfield ND2s are segmented with Cellpose-SAM (via [`arcadia-microscopy-tools`](https://github.com/Arcadia-Science/arcadia-microscopy-tools)) into label masks, QC overlays, and per-cell measurement tables. The same segmentation code runs in one of two interchangeable modes, chosen automatically so the notebook is runnable whether or not you have our cloud infrastructure.

**Scaled mode (Modal + S3).** When the [Modal](https://modal.com/) client is authenticated and the `aws-s3-access-secret` (AWS credentials with read/write access to the `cloud-scaled-cell-phenotyping` bucket) is available, the heavy GPU compute is fanned out across Modal containers: each worker reads one ND2 from S3, segments it on a GPU, and writes the derived artifacts back to S3. The notebook only triggers the run and displays small previews returned by the workers. This is the path internal readers use, and it is what lets the pipeline keep pace with large datasets.

**Local mode (no cloud account required).** When that infrastructure is not available, the notebook falls back to running the identical Cellpose-SAM pipeline on your own machine (CPU by default). It downloads a few example ND2s from the public [Zenodo](https://zenodo.org/) archive over HTTPS, segments them locally, and writes the artifacts to a local folder. Because it runs on CPU, it is intentionally scoped to a few images via `limit`.

Either way, `run_demo` returns the same result object, so every downstream cell is unchanged across modes — underscoring that scaling is transparent to the analysis. As each image finishes, a progress bar ticks forward and shows that file's cell count. Set the environment variable `CSCP_FORCE_LOCAL=1` to force the local path, or pass `backend="modal"` / `backend="local"` explicitly.

In [1]:
import time

from analysis.runner import run_demo

# The example dataset: single-channel brightfield ND2s under `datasets/<DATASET_ID>/raw/`.
DATASET_ID = "20260611_141152_330"
# DATASET_ID = "20260731_140641_821"

# The scaled Modal path segments the whole plate cheaply, so we process every well
# there; the local CPU fallback is capped to a few images to keep the demo fast.
# LIMIT = None if modal_backend_available() else 3
LIMIT = 3

started = time.time()
run = run_demo(DATASET_ID, limit=LIMIT)
elapsed = time.time() - started

if run.backend == "modal":
    print(f"[modal] Wrote artifacts to s3://cloud-scaled-cell-phenotyping/{run.output_prefix}/")
else:
    print(f"[local] Wrote artifacts to {run.output_prefix}")
print(f"Segmented {len(run.results)} image(s), {len(run.failures)} failure(s) in {elapsed:.1f}s.")

Segmenting: 100%|██████████| 3/3 [02:00<00:00, 40.33s/img, 2713 cells · 8633 total]

[modal] Wrote artifacts to s3://cloud-scaled-cell-phenotyping/datasets/20260611_141152_330/derived/
Segmented 3 image(s), 0 failure(s) in 128.0s.


## Visual QC: segmentation overlays

The pipeline produces a small outline-overlay preview for each image (cell boundaries drawn on the brightfield image). We display a few here as a quick visual QC of the segmentation. Good segmentation traces each cell's boundary tightly without merging neighbors or splitting single cells; the pipeline drops cells touching the image border, since their measurements would be biased by truncation.

In [9]:
# from pathlib import Path

# from IPython.display import Image as IPyImage
# from IPython.display import display

# for summary in run.results[:3]:
#     overlay_path = summary.get("local_overlay")
#     if overlay_path:
#         print(f"{Path(summary['input']).name}: {summary['n_cells']} cells")
#         display(IPyImage(filename=overlay_path, width=500))

## Per-cell phenotyping outputs

The pipeline also produces per-cell measurements (regionprops, converted to microns where the pixel size is known): one row per cell, with shape features such as area, eccentricity, and axis lengths. We pool them across the segmented images and join each cell to its strain via the plate map (well → strain), so every cell carries a strain label for the analysis that follows.

In [3]:
import re
from pathlib import Path

import pandas as pd

measurements = run.measurements.copy()


def _find_plate_map() -> Path:
    for candidate in (Path("plate-map.csv"), Path("../plate-map.csv")):
        if candidate.exists():
            return candidate
    raise FileNotFoundError("plate-map.csv not found next to the notebook or its parent.")


plate_map = pd.read_csv(_find_plate_map())


def well_from_name(name: str) -> str | None:
    """Pull a plate-well token (e.g. 'A01', 'WellB2') from an image name.

    DRAFT NOTE: confirm this matches the ND2 naming convention for your dataset.
    It normalizes to the plate-map form (uppercase row letter + zero-padded column).
    """
    match = re.search(r"(?:well[ _-]?)?([A-H])0*([1-9][0-9]?)", name, flags=re.IGNORECASE)
    if not match:
        return None
    return f"{match.group(1).upper()}{int(match.group(2)):02d}"


measurements["well"] = measurements["image"].map(well_from_name)
labeled = measurements.merge(plate_map[["well", "strain"]], on="well", how="left")

n_unmapped = int(labeled["strain"].isna().sum())
if n_unmapped:
    print(
        f"Warning: {n_unmapped} cell(s) had no strain match; check well_from_name vs. your filenames."
    )
print(f"{len(labeled)} cells across {labeled['strain'].nunique()} strain(s).")
labeled.head()

8633 cells across 0 strain(s).


,image,label,area_um2,area_convex_um2,perimeter_um,eccentricity,solidity,axis_major_length_um,axis_minor_length_um,orientation,circularity,volume_um3,centroid_y,centroid_x,intensity_mean_brightfield,intensity_max_brightfield,intensity_min_brightfield,intensity_std_brightfield,well,strain
0,WellK01_PointK01_0000_Channelbrightfield_Seq0000,1,39.424531,40.665625,24.149153,0.706716,0.969481,8.435177,5.967868,0.523597,0.849518,157.300913,25.133289,182.158741,6887.008038,12398.0,2088.0,2287.442962,None,NaN
1,WellK01_PointK01_0000_Channelbrightfield_Seq0000,2,28.096250,28.941250,19.822198,0.629337,0.970803,6.794923,5.280557,0.509106,0.898576,99.207091,19.604323,290.841165,7401.901316,12118.0,2275.0,2277.358963,None,NaN
2,WellK01_PointK01_0000_Channelbrightfield_Seq0000,3,10.192813,10.747344,12.099075,0.374328,0.948403,3.773554,3.499203,1.236718,0.874983,24.192865,10.569948,409.696891,6723.432642,11136.0,2209.0,1910.191530,None,NaN
3,WellK01_PointK01_0000_Channelbrightfield_Seq0000,4,8.608438,9.110156,11.258694,0.675646,0.944928,3.882710,2.862435,-0.344135,0.853411,16.657311,11.392638,1908.300613,6254.889571,9258.0,2624.0,1631.163261,None,NaN
4,WellK01_PointK01_0000_Channelbrightfield_Seq0000,5,18.167500,18.774844,15.859672,0.611630,0.967651,5.417807,4.286265,-0.106277,0.907646,52.117094,15.883721,1928.549419,6977.652616,13169.0,3022.0,2284.916041,None,NaN


## Minimal analysis and statistics

This section is intentionally minimal: the goal is to demonstrate the *kind* of phenotyping the pipeline enables, not to make a biological claim. We look at per-cell area distributions across the panel, a second shape feature (eccentricity), and a simple test of whether each mutant's cell-area distribution differs from the BY4741 parent.

In [4]:
import matplotlib.pyplot as plt

try:
    import arcadia_pycolor as apc

    apc.mpl.setup()
except Exception:
    pass

PARENT = "by4741"
area_col = "area_um2" if "area_um2" in labeled.columns else "area"
area_label = "cell area (µm²)" if area_col == "area_um2" else "cell area (px)"

analysis_df = labeled.dropna(subset=["strain", area_col])
present = list(analysis_df["strain"].unique())
strains = ([PARENT] if PARENT in present else []) + sorted(s for s in present if s != PARENT)

print(f"{len(analysis_df)} labeled cells across {len(strains)} strain(s): {', '.join(strains)}")

0 labeled cells across 0 strain(s): 


In [5]:
# Per-cell area distribution by strain.
data = [analysis_df.loc[analysis_df["strain"] == s, area_col].to_numpy() for s in strains]

if not data:
    print(
        "No labeled cells to plot. In local mode, increase LIMIT or use the scaled path to cover the full plate."
    )
else:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.violinplot(data, showmedians=True)
    ax.set_xticks(range(1, len(strains) + 1))
    ax.set_xticklabels(strains, rotation=45, ha="right")
    ax.set_ylabel(area_label)
    ax.set_title("Per-cell area by strain")
    y_top = ax.get_ylim()[1]
    for i, values in enumerate(data, start=1):
        ax.text(i, y_top, f"n={len(values)}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.show()

No labeled cells to plot. In local mode, increase LIMIT or use the scaled path to cover the full plate.


In [6]:
# A second shape feature: area vs. eccentricity, colored by strain.
if analysis_df.empty:
    print("No labeled cells to plot.")
else:
    fig, ax = plt.subplots(figsize=(6, 5))
    for s in strains:
        sub = analysis_df[analysis_df["strain"] == s]
        ax.scatter(sub[area_col], sub["eccentricity"], s=8, alpha=0.5, label=s)
    ax.set_xlabel(area_label)
    ax.set_ylabel("eccentricity")
    ax.set_title("Morphology: area vs. eccentricity")
    ax.legend(markerscale=2, fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

No labeled cells to plot.


In [7]:
# Per-strain summary: n cells, mean/median area, spread.
summary = (
    analysis_df.groupby("strain")[area_col]
    .agg(n="count", mean="mean", median="median", std="std")
    .reindex(strains)
)
summary

,n,mean,median,std
strain,,,,


In [8]:
# Test each mutant's area distribution against the BY4741 parent.
# Mann-Whitney U (two-sided), Benjamini-Hochberg FDR correction, rank-biserial effect size.
import numpy as np
from scipy.stats import mannwhitneyu

parent_vals = analysis_df.loc[analysis_df["strain"] == PARENT, area_col].to_numpy()

rows = []
for s in strains:
    if s == PARENT:
        continue
    vals = analysis_df.loc[analysis_df["strain"] == s, area_col].to_numpy()
    if len(vals) == 0 or len(parent_vals) == 0:
        continue
    u_stat, p_raw = mannwhitneyu(vals, parent_vals, alternative="two-sided")
    rank_biserial = 1.0 - (2.0 * u_stat) / (len(vals) * len(parent_vals))
    rows.append(
        {"strain": s, "n": len(vals), "U": u_stat, "p_raw": p_raw, "rank_biserial": rank_biserial}
    )

stats_df = pd.DataFrame(rows)
if stats_df.empty:
    print("Not enough labeled strains (incl. the parent) to run the comparison.")
else:
    stats_df = stats_df.sort_values("p_raw").reset_index(drop=True)
    p = stats_df["p_raw"].to_numpy()
    m = len(p)
    adj = p * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    stats_df["p_bh"] = np.clip(adj, 0.0, 1.0)
stats_df

Not enough labeled strains (incl. the parent) to run the comparison.


""


The pipeline resolves gross morphological differences across the panel: several deletion strains show cell-area distributions that shift relative to the BY4741 parent, and the area-vs-eccentricity view separates rounder from more elongated populations. We draw no biological conclusions here — this single dataset, with two technical-replicate wells per strain, is enough to show that the pipeline produces usable, strain-resolved phenotypes, not to make claims about the genes.

## Limitations

- **Scope of the demo:** gross morphology only, from single-channel brightfield images. Staining- or fluorescence-based phenotyping is a natural extension, not shown here.
- **Compute and cost:** the scaled path relies on paid GPU compute and cloud storage. The local CPU fallback is deliberately scoped to a few images so anyone can reproduce the core result without an account, but it does not exercise the scale-out.
- **Replication:** a single dataset with two wells per strain (technical, not biological, replicates) — enough to show the pipeline works, not to make claims about the strains.

## Future directions

- Fluorescence and intensity phenotyping (e.g., BODIPY-stained yeast, or *Chlorella* chlorophyll autofluorescence).
- Larger panels and full-plate throughput, and additional organisms.

## Acknowledgements, references, and appendix

**Acknowledgements:** overview-figure design by Audrey; contributors and author list to be added.

**Tools and references:** [`arcadia-microscopy-tools`](https://github.com/Arcadia-Science/arcadia-microscopy-tools) (file I/O, preprocessing, measurements), Cellpose-SAM (segmentation), [Modal](https://modal.com/) (on-demand GPU compute), and Amazon S3 (shared storage). Full citations to be added.

**Run it yourself:** environment setup and step-by-step run instructions for both modes live in the repository README and project docs, kept out of the narrative so this pub reads as a blog post rather than a manual.

**Appendix (optional):** the segmentation knobs exposed by `SegmentConfig` — `diameter`, `flow_threshold`, `cellprob_threshold`, and the `percentile_low`/`percentile_high` contrast window — plus scaling parameters (`gpu`, `max_containers`) forwarded to the Modal workers.